![Math Wonders](images_store/KEEP/mathwonders.png)

# Tensor Art Gallery - Mathematical Visualizations with Candle

This notebook generates beautiful mathematical visualizations using Candle tensor operations and compile-time expression parsing. We create artistic patterns ranging from simple geometric forms to complex fractals and interference patterns.

**Features:**
- High-resolution coordinate grids (up to 1024x1024)
- Expression parser for compile-time mathematical evaluation
- Advanced pattern generation (fractals, waves, gradients)
- Efficient tensor-to-image conversion
- Performance benchmarking and optimization
- Interactive parameter exploration

**Mathematical Patterns:**
- $f(x,y) = \sin(ax) \cos(by)$ - Wave interference
- $g(x,y) = e^{-((x-x_0)^2 + (y-y_0)^2)/(2\sigma^2)}$ - Gaussian distributions
- $h(x,y) = \sin(\sqrt{x^2 + y^2} \cdot k)$ - Radial waves
- $m(x,y) = |\sin(x/s) + \sin(y/s)|$ - Complex interference patterns

In [36]:
:toolchain stable
:dep candle-notebooks = "0.1.0" 
// central  re-export

use candle_notebooks as nb;
use nb::*;
use std::fs;
let device = nb::Device::Cpu;
nb::set_notebook_cwd().expect("failed to set notebook cwd");
fs::create_dir_all("images_store").ok();
println!("Toolchain: stable | Device: {:?} | CWD: {}", device, std::env::current_dir().unwrap().display());

Notebook CWD set to repository root: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks
Toolchain: stable | Device: Cpu | CWD: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks


Toolchain: stable


In [37]:
#![allow(non_snake_case)]

// Import required modules
use candle_notebooks::{Tensor, DType, Device, D};
use candle_notebooks::ah as anyhow; 
use anyhow::Result;
use std::f64::consts::{PI, E};
use std::time::Instant;
use std::collections::HashMap;
use candle_notebooks as nb;

// Helper macro: run a Result block in a cell without closures
#[allow(unused_macros)]
macro_rules! try_cell {
    ($body:block) => {{
        let __res: Result<()> = (|| -> Result<()> { $body })();
        if let Err(e) = __res {
            panic!("cell failed: {}", e);
        }
    }};
}

println!("✓ Dependencies loaded: candle-notebooks (anyhow via helper), image, base64 (via helper)");

// Initialize workspace
nb::set_notebook_cwd().unwrap();
nb::set_image_store_rel_dir("images_store").unwrap();
std::fs::create_dir_all("images_store").ok();

println!("✓ Notebook initialized successfully!");

✓ Dependencies loaded: candle-notebooks (anyhow via helper), image, base64 (via helper)
Notebook CWD set to repository root: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks
✓ Notebook initialized successfully!


Saving images to: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks/images_store

In [38]:
// Coordinate Grid Generation - High Resolution Support
// Efficient grid creation with broadcasting for complex mathematical operations

fn create_coordinate_grid(width: usize, height: usize, x_range: (f64, f64), y_range: (f64, f64), device: &Device) -> Result<(Tensor, Tensor)> {
    let (x_min, x_max) = x_range;
    let (y_min, y_max) = y_range;
    
    // Create coordinate vectors
    let xs: Vec<f64> = (0..width)
        .map(|i| x_min + (x_max - x_min) * (i as f64) / (width as f64 - 1.0))
        .collect();
    let ys: Vec<f64> = (0..height)
        .map(|j| y_min + (y_max - y_min) * (j as f64) / (height as f64 - 1.0))
        .collect();
    
    // Convert to tensors and prepare for broadcasting
    let x_tensor = Tensor::new(xs, device)?.reshape((width, 1))?;
    let y_tensor = Tensor::new(ys, device)?.reshape((1, height))?;
    
    // Broadcast to full grid
    let x_grid = x_tensor.broadcast_as((width, height))?;
    let y_grid = y_tensor.broadcast_as((width, height))?;
    
    Ok((x_grid, y_grid))
}

In [39]:
// Create multiple resolution grids for different use cases
let (width, height): (usize, usize) = (512, 512);  // High resolution for detailed patterns



let (x_var, y_var) = create_coordinate_grid(width, height, (-2.0, 2.0), (-2.0, 2.0), &device)
    .expect("create_coordinate_grid failed");

println!("• Coordinate grids created: {}x{}", width, height);
println!("Grid memory: ~{:.1} MB", (width * height * 8 * 2) as f64 / 1_000_000.0);
println!("Range: X ∈ [-2, 2], Y ∈ [-2, 2]");

• Coordinate grids created: 512x512
Grid memory: ~4.2 MB
Range: X ∈ [-2, 2], Y ∈ [-2, 2]
Grid memory: ~4.2 MB
Range: X ∈ [-2, 2], Y ∈ [-2, 2]


In [40]:
// Basic Mathematical Functions - Foundation Patterns
// Generate fundamental mathematical visualizations

try_cell!({
    fn compute_stats(tensor: &Tensor, name: &str) -> Result<()> {
        let flat = tensor.flatten_all()?;
        let min_val = flat.min(0)?.to_scalar::<f64>()?;
        let max_val = flat.max(0)?.to_scalar::<f64>()?;
        println!("{}: range [{:.3}, {:.3}]", name, min_val, max_val);
        Ok(())
    }

    // 1. Wave Interference Pattern: sin(ax) * cos(by)
    let a: f64 = 8.0; // X frequency
    let b: f64 = 6.0; // Y frequency
    let wave_x = (&x_var * a)?.sin()?;
    let wave_y = (&y_var * b)?.cos()?;
    let interference = (&wave_x * &wave_y)?;
    compute_stats(&interference, "🌊 Wave Interference")?;

    // 2. Radial Wave Pattern: sin(r * k) where r = sqrt(x² + y²)
    let k: f64 = 10.0; // Radial frequency
    let x2 = (&x_var * &x_var)?;
    let y2 = (&y_var * &y_var)?;
    let r_squared = (&x2 + &y2)?;
    let r = r_squared.sqrt()?;
    let radial_waves = (&r * k)?.sin()?;
    compute_stats(&radial_waves, "🎯 Radial Waves")?;

    // 3. Gaussian Distribution: exp(-r²/(2σ²))
    let sigma: f64 = 0.8;
    let gaussian_arg = (&r_squared / (2.0 * sigma * sigma))?;
    let gaussian = (gaussian_arg * -1.0)?.exp()?;
    compute_stats(&gaussian, "🔔 Gaussian Distribution")?;

    // 4. Complex Spiral: sin(r) * cos(5*theta) - using approximation for polar angle
    // Since atan2 isn't available, we'll create a different spiral pattern
    let spiral_r = (&r * 5.0)?.sin()?;
    let spiral_theta = (&x_var * 8.0)?.cos()?; // Alternative spiral using X coordinate
    let spiral = (&spiral_r * &spiral_theta)?;
    compute_stats(&spiral, "🌀 Complex Spiral")?;

    println!("✨ Basic mathematical patterns generated successfully!");
    Ok(())
});

🌊 Wave Interference: range [-1.000, 1.000]
🎯 Radial Waves: range [-1.000, 1.000]
🔔 Gaussian Distribution: range [0.002, 1.000]
🌀 Complex Spiral: range [-1.000, 1.000]
✨ Basic mathematical patterns generated successfully!


In [41]:
// Expression Parser Integration
// Compile-time mathematical expression evaluation with validation

use candle_notebooks::expr::{ExprEnv, eval_expr};

// Create expression environment with parameters
let mut params: std::collections::HashMap<String, f64> = std::collections::HashMap::new();
params.insert("a".to_string(), 8.0);
params.insert("b".to_string(), 6.0);
params.insert("k".to_string(), 10.0);
params.insert("sigma".to_string(), 0.8);
params.insert("pi".to_string(), PI);
params.insert("e".to_string(), E);

// Convert coordinates to F64 for expression parser
let env = ExprEnv::new(
    x_var.clone().to_dtype(DType::F64)?, 
    y_var.clone().to_dtype(DType::F64)?, 
    params
)?;

println!("🧮 Expression parser environment ready!");
println!("Available variables: x, y, a, b, k, sigma, pi, e");

// Function to evaluate and time expressions
fn eval_and_time(expr: &str, env: &ExprEnv, description: &str) -> Result<Tensor> {
    let start = Instant::now();
    let result = eval_expr(expr, env)?;
    let elapsed = start.elapsed();
    
    let flat = result.flatten_all()?;
    let min_val = flat.min(0)?.to_scalar::<f32>()?;
    let max_val = flat.max(0)?.to_scalar::<f32>()?;
    
    println!("{}: range [{:.3}, {:.3}] | {:.2}ms", 
        description, min_val, max_val, elapsed.as_secs_f64() * 1000.0);
    println!("  Expression: {}", expr);
    
    Ok(result)
}

// Generate patterns using expression parser
let expr_interference = eval_and_time(
    "sin(a*x) * cos(b*y)", 
    &env, 
    "🎼 Parsed Wave Interference"
)?;

let expr_radial = eval_and_time(
    "sin(sqrt(x*x + y*y) * k)", 
    &env, 
    "🎯 Parsed Radial Waves"
)?;

let expr_gaussian = eval_and_time(
    "exp(-(x*x + y*y)/(2*sigma*sigma))", 
    &env, 
    "🔔 Parsed Gaussian"
)?;

println!("✅ Expression parsing validation complete!");

🧮 Expression parser environment ready!
Available variables: x, y, a, b, k, sigma, pi, e
Available variables: x, y, a, b, k, sigma, pi, e
🎼 Parsed Wave Interference: range [-1.000, 1.000] | 11.75ms
  Expression: sin(a*x) * cos(b*y)
🎯 Parsed Radial Waves: range [-1.000, 1.000] | 12.48ms
  Expression: sin(sqrt(x*x + y*y) * k)
🔔 Parsed Gaussian: range [0.002, 1.000] | 12.38ms
  Expression: exp(-(x*x + y*y)/(2*sigma*sigma))
✅ Expression parsing validation complete!


In [42]:
// Advanced Pattern Generation
// Complex mathematical visualizations using both tensor ops and expressions

// 1. Mandelbrot-inspired Pattern
let mandel_expr = eval_and_time(
    "sin(x*x - y*y + x) * cos(2*x*y + y)",
    &env,
    "🌌 Mandelbrot-inspired Pattern"
)?;

// 2. Wave Interference with Phase Shift
let phase_waves = eval_and_time(
    "sin(a*x + pi/4) * cos(b*y) + 0.5*sin(b*x) * cos(a*y + pi/2)",
    &env,
    "🌊 Phase-shifted Interference"
)?;

// 3. Checkerboard with Smooth Transitions
let smooth_checker = eval_and_time(
    "sin(10*x) * sin(10*y) * exp(-(x*x + y*y)/2)",
    &env,
    "🏁 Smooth Checkerboard"
)?;

// 4. Flower Pattern (using alternative to atan2)
let flower = eval_and_time(
    "sin(6*sqrt(x*x + y*y)) * cos(8*x/(sqrt(x*x + y*y) + 0.1))",
    &env,
    "🌸 Flower Pattern"
)?;

// 5. Heat Equation Solution
let heat_pattern = eval_and_time(
    "exp(-((x-0.5)*(x-0.5) + (y-0.5)*(y-0.5))/0.3) + exp(-((x+0.5)*(x+0.5) + (y+0.5)*(y+0.5))/0.3)",
    &env,
    "🔥 Heat Equation Pattern"
)?;

// 6. Gradient Field Visualization
let gradient_field = eval_and_time(
    "sin(x*y) + cos(x*x - y*y) * 0.5",
    &env,
    "📈 Gradient Field"
)?;

// 7. Turbulence Pattern
let turbulence = eval_and_time(
    "sin(4*x + 2*sin(3*y)) * cos(4*y + 2*cos(3*x))",
    &env,
    "🌪️ Turbulence Pattern"
)?;

println!("🎨 Advanced pattern gallery complete!");
println!("Generated {} unique mathematical visualizations", 7);

🌌 Mandelbrot-inspired Pattern: range [-1.000, 1.000] | 12.46ms
  Expression: sin(x*x - y*y + x) * cos(2*x*y + y)
  Expression: sin(x*x - y*y + x) * cos(2*x*y + y)
🌊 Phase-shifted Interference: range [-1.456, 1.456] | 26.40ms
  Expression: sin(a*x + pi/4) * cos(b*y) + 0.5*sin(b*x) * cos(a*y + pi/2)
🏁 Smooth Checkerboard: range [-0.975, 0.975] | 18.13ms
  Expression: sin(10*x) * sin(10*y) * exp(-(x*x + y*y)/2)
🌸 Flower Pattern: range [-1.000, 1.000] | 22.21ms
  Expression: sin(6*sqrt(x*x + y*y)) * cos(8*x/(sqrt(x*x + y*y) + 0.1))
🔥 Heat Equation Pattern: range [0.000, 1.001] | 24.60ms
  Expression: exp(-((x-0.5)*(x-0.5) + (y-0.5)*(y-0.5))/0.3) + exp(-((x+0.5)*(x+0.5) + (y+0.5)*(y+0.5))/0.3)
📈 Gradient Field: range [-1.500, 1.500] | 13.96ms
  Expression: sin(x*y) + cos(x*x - y*y) * 0.5
🌪️ Turbulence Pattern: range [-1.000, 1.000] | 24.28ms
  Expression: sin(4*x + 2*sin(3*y)) * cos(4*y + 2*cos(3*x))
🎨 Advanced pattern gallery complete!
Generated 7 unique mathematical visualizations


In [45]:
// Image Normalization and Export
// Efficient tensor-to-image conversion with edge case handling

use image::{ImageBuffer, Rgb, RgbImage};

fn tensor_to_image(tensor: &Tensor, name: &str) -> Result<Vec<u8>> {
    let flat = tensor.flatten_all()?;
    let min_val = flat.min(0)?.to_scalar::<f32>()?;
    let max_val = flat.max(0)?.to_scalar::<f32>()?;
    let span = max_val - min_val;
    
    println!("🖼️ Converting '{}': range [{:.3}, {:.3}]", name, min_val, max_val);
    
    // Normalize to [0, 1] with edge case handling
    let normalized = if span.abs() < 1e-8 {
        // Constant tensor - map to middle gray
        println!("  ⚠️ Constant tensor detected, using middle gray");
        (tensor.zeros_like()? + 0.5)?
    } else {
        // Proper normalization: (tensor - min) / span
        let shifted = (tensor - min_val as f64)?;
        (shifted / span as f64)?
    };
    
    // Scale to [0, 255] and convert to u8
    let scaled = (normalized * 255.0)?;
    let data = scaled.flatten_all()?.to_vec1::<f32>()?;
    let u8_data: Vec<u8> = data.into_iter()
        .map(|v| v.round().clamp(0.0, 255.0) as u8)
        .collect();
    
    println!("  ✅ Conversion complete: {} pixels", u8_data.len());
    Ok(u8_data)
}

fn create_rgb_image(data: &[u8], width: usize, height: usize, colormap: &str) -> RgbImage {
    let mut img = ImageBuffer::new(width as u32, height as u32);
    
    for (i, &intensity) in data.iter().enumerate() {
        let x = (i % width) as u32;
        let y = (i / width) as u32;
        
        let rgb = match colormap {
            "grayscale" => [intensity, intensity, intensity],
            "heat" => {
                // Heat map: black -> red -> yellow -> white
                let t = intensity as f32 / 255.0;
                let r = (255.0 * t.min(1.0)) as u8;
                let g = (255.0 * (2.0 * t - 1.0).max(0.0).min(1.0)) as u8;
                let b = (255.0 * (3.0 * t - 2.0).max(0.0).min(1.0)) as u8;
                [r, g, b]
            },
            "ocean" => {
                // Ocean map: dark blue -> cyan -> yellow
                let t = intensity as f32 / 255.0;
                let r = (255.0 * (2.0 * t - 1.0).max(0.0)) as u8;
                let g = (255.0 * t) as u8;
                let b = (255.0 * (1.0 - 0.5 * t)) as u8;
                [r, g, b]
            },
            _ => [intensity, intensity, intensity], // Default to grayscale
        };
        
        img.put_pixel(x, y, Rgb(rgb));
    }
    
    img
}

// Convert patterns to images
let interference_data: Vec<u8> = tensor_to_image(&expr_interference, "Wave Interference").expect("tensor_to_image failed (interference)");
let radial_data: Vec<u8> = tensor_to_image(&expr_radial, "Radial Waves").expect("tensor_to_image failed (radial)");
let mandel_data: Vec<u8> = tensor_to_image(&mandel_expr, "Mandelbrot Pattern").expect("tensor_to_image failed (mandelbrot)");
let flower_data: Vec<u8> = tensor_to_image(&flower, "Flower Pattern").expect("tensor_to_image failed (flower)");

println!("🎨 Image conversion complete!");
println!("Ready for visualization and export");

🖼️ Converting 'Wave Interference': range [-1.000, 1.000]
  ✅ Conversion complete: 262144 pixels
🖼️ Converting 'Radial Waves': range [-1.000, 1.000]
  ✅ Conversion complete: 262144 pixels
🖼️ Converting 'Mandelbrot Pattern': range [-1.000, 1.000]
  ✅ Conversion complete: 262144 pixels
🖼️ Converting 'Flower Pattern': range [-1.000, 1.000]
  ✅ Conversion complete: 262144 pixels
🎨 Image conversion complete!
Ready for visualization and export
  ✅ Conversion complete: 262144 pixels
🖼️ Converting 'Radial Waves': range [-1.000, 1.000]
  ✅ Conversion complete: 262144 pixels
🖼️ Converting 'Mandelbrot Pattern': range [-1.000, 1.000]
  ✅ Conversion complete: 262144 pixels
🖼️ Converting 'Flower Pattern': range [-1.000, 1.000]
  ✅ Conversion complete: 262144 pixels
🎨 Image conversion complete!
Ready for visualization and export


In [47]:
// Interactive Parameter Exploration
// Dynamic mathematical visualization with parameter sweeps

try_cell!({
    fn parameter_sweep(base_expr: &str, param_name: &str, values: &[f64], env_template: &ExprEnv) -> Result<Vec<Tensor>> {
        println!("🔄 Parameter sweep: {} = {:?}", param_name, values);
        let mut results = Vec::new();
        
        for &value in values {
            // Create new environment with updated parameter
            let mut params = env_template.params.clone();
            params.insert(param_name.to_string(), value);
            
            let sweep_env = ExprEnv::new(
                env_template.x.clone(),
                env_template.y.clone(),
                params
)?;
            
            let result = eval_expr(base_expr, &sweep_env)?;
            results.push(result);
            
            print!(".");
        }
        
        println!(" ✅ Generated {} variations", results.len());
        Ok(results)
    }

    // 1. Frequency Sweep for Wave Patterns
    let freq_values = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0];
    let wave_variations: Vec<Tensor> = parameter_sweep(
        "sin(a*x) * cos(a*y)", 
        "a", 
        &freq_values, 
        &env
    )?;

    // 2. Sigma Sweep for Gaussian Distributions
    let sigma_values = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2];
    let gaussian_variations: Vec<Tensor> = parameter_sweep(
        "exp(-(x*x + y*y)/(2*sigma*sigma))", 
        "sigma", 
        &sigma_values, 
        &env
    )?;

    // 3. Complex Pattern Evolution (alternative to atan2)
    let evolution_expr = "sin(a*sqrt(x*x + y*y)) * cos(a*x/(sqrt(x*x + y*y) + 0.1))";
    let spiral_values = [3.0, 6.0, 9.0, 12.0, 15.0, 18.0];
    let spiral_evolution: Vec<Tensor> = parameter_sweep(
        evolution_expr,
        "a",
        &spiral_values,
        &env
    )?;

    // Calculate statistics for each variation
    for (i, tensor) in wave_variations.iter().enumerate() {
        let flat = tensor.flatten_all()?;
        let mean = flat.sum_all()?.to_scalar::<f32>()? / (width * height) as f32;
        println!("Wave freq={:.1}: mean={:.3}", freq_values[i], mean);
    }

    println!("🎭 Interactive parameter exploration complete!");
    println!("Generated {} parameter sweeps with {} total variations", 3, wave_variations.len() + gaussian_variations.len() + spiral_evolution.len());
    Ok(())
});

🔄 Parameter sweep: a = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0]
...... ✅ Generated 6 variations
🔄 Parameter sweep: sigma = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
...... ✅ Generated 6 variations
🔄 Parameter sweep: a = [3.0, 6.0, 9.0, 12.0, 15.0, 18.0]
...... ✅ Generated 6 variations
Wave freq=2.0: mean=0.000
Wave freq=4.0: mean=0.000
Wave freq=6.0: mean=-0.000
Wave freq=8.0: mean=-0.000
Wave freq=10.0: mean=-0.000
Wave freq=12.0: mean=0.000
🎭 Interactive parameter exploration complete!
Generated 3 parameter sweeps with 18 total variations


In [48]:
// Complex Mathematical Visualizations
// Sophisticated mathematical art and advanced pattern generation

// 1. Fractal-inspired Patterns
let julia_set = eval_and_time(
    "sin(x*x - y*y + 0.7*x) * cos(2*x*y + 0.3*y)",
    &env,
    "🌀 Julia Set Approximation"
)?;

// 2. Wave Equation Solution
let wave_equation = eval_and_time(
    "sin(sqrt((x-0.5)*(x-0.5) + y*y)*10) + sin(sqrt((x+0.5)*(x+0.5) + y*y)*10)",
    &env,
    "🌊 Wave Equation (Two Sources)"
)?;

// 3. Electromagnetic Field Visualization
let em_field = eval_and_time(
    "cos(5*x)*exp(-y*y) + sin(5*y)*exp(-x*x)",
    &env,
    "⚡ Electromagnetic Field"
)?;

// 4. Crystalline Structure
let crystal = eval_and_time(
    "cos(6*x)*cos(6*y) + cos(6*x)*cos(6*(x+y)) + cos(6*y)*cos(6*(x+y))",
    &env,
    "💎 Crystalline Structure"
)?;

// 5. Plasma Physics Simulation
let plasma = eval_and_time(
    "sin(3*x + 2*sin(4*y)) * cos(3*y + 2*cos(4*x)) * exp(-(x*x + y*y)/4)",
    &env,
    "🌩️ Plasma Physics"
)?;

// 6. Quantum Wavefunction
let quantum = eval_and_time(
    "exp(-(x*x + y*y)/2) * cos(5*x*y) * sin(3*sqrt(x*x + y*y))",
    &env,
    "⚛️ Quantum Wavefunction"
)?;

// 7. Topological Map
let topology = eval_and_time(
    "sin(x*x + y*y) + 0.5*sin(2*(x*x + y*y)) + 0.25*sin(4*(x*x + y*y))",
    &env,
    "🗺️ Topological Map"
)?;

// Advanced composition: Multiple overlays
let composition_expr = "0.4*sin(a*x)*cos(b*y) + 0.3*exp(-(x*x + y*y)/(2*sigma*sigma)) + 0.3*sin(k*sqrt(x*x + y*y))";
let masterpiece = eval_and_time(
    composition_expr,
    &env,
    "🎨 Mathematical Masterpiece"
)?;

println!("🎭 Complex mathematical art gallery complete!");
println!("Created {} sophisticated visualizations", 8);
println!("Peak mathematical expression complexity: {} characters", composition_expr.len());

🌀 Julia Set Approximation: range [-1.000, 1.000] | 14.81ms
  Expression: sin(x*x - y*y + 0.7*x) * cos(2*x*y + 0.3*y)
  Expression: sin(x*x - y*y + 0.7*x) * cos(2*x*y + 0.3*y)
🌊 Wave Equation (Two Sources): range [-2.000, 2.000] | 22.20ms
  Expression: sin(sqrt((x-0.5)*(x-0.5) + y*y)*10) + sin(sqrt((x+0.5)*(x+0.5) + y*y)*10)
⚡ Electromagnetic Field: range [-1.604, 1.912] | 13.98ms
  Expression: cos(5*x)*exp(-y*y) + sin(5*y)*exp(-x*x)
💎 Crystalline Structure: range [-1.000, 3.000] | 27.36ms
  Expression: cos(6*x)*cos(6*y) + cos(6*x)*cos(6*(x+y)) + cos(6*y)*cos(6*(x+y))
🌩️ Plasma Physics: range [-0.965, 0.959] | 27.94ms
  Expression: sin(3*x + 2*sin(4*y)) * cos(3*y + 2*cos(4*x)) * exp(-(x*x + y*y)/4)
⚛️ Quantum Wavefunction: range [-0.328, 0.884] | 20.83ms
  Expression: exp(-(x*x + y*y)/2) * cos(5*x*y) * sin(3*sqrt(x*x + y*y))
🗺️ Topological Map: range [-1.221, 1.221] | 25.18ms
  Expression: sin(x*x + y*y) + 0.5*sin(2*(x*x + y*y)) + 0.25*sin(4*(x*x + y*y))
🎨 Mathematical Masterpiece: rang

In [49]:
// Performance Benchmarking
// Compare tensor operations vs expression parsing performance

use std::time::{Duration, Instant};

try_cell!({
    fn benchmark_tensor_ops(iterations: usize, env: &ExprEnv) -> Result<Duration> {
        let start = Instant::now();
        
        for _ in 0..iterations {
            // Manual tensor operations using coordinates from environment
            let xa = (&env.x * 8.0)?;
            let yb = (&env.y * 6.0)?;
            let _result = (xa.sin()? * yb.cos()?)?;
        }
        
        Ok(start.elapsed())
    }

    fn benchmark_expression_parsing(iterations: usize, env: &ExprEnv) -> Result<Duration> {
        let start = Instant::now();
        
        for _ in 0..iterations {
            let _result = eval_expr("sin(a*x) * cos(b*y)", env)?;
        }
        
        Ok(start.elapsed())
    }

    fn benchmark_complex_expression(iterations: usize, env: &ExprEnv) -> Result<Duration> {
        let start = Instant::now();
        let complex_expr = "sin(a*x + pi/4) * cos(b*y) + 0.5*exp(-(x*x + y*y)/(2*sigma*sigma))";
        
        for _ in 0..iterations {
            let _result = eval_expr(complex_expr, env)?;
        }
        
        Ok(start.elapsed())
    }

    println!("⚡ Performance Benchmarking Suite");
    println!("Grid size: {}x{} = {} pixels", width, height, width * height);
    println!("");

    let iterations = 10;
    println!("Running {} iterations of each benchmark...", iterations);

    // Benchmark tensor operations
    let tensor_time = benchmark_tensor_ops(iterations, &env)?;
    let tensor_avg = tensor_time.as_secs_f64() / iterations as f64;
    println!("🔧 Manual Tensor Ops: {:.2}ms avg ({:.1}ms total)", 
        tensor_avg * 1000.0, tensor_time.as_secs_f64() * 1000.0);

    // Benchmark expression parsing
    let expr_time = benchmark_expression_parsing(iterations, &env)?;
    let expr_avg = expr_time.as_secs_f64() / iterations as f64;
    println!("📝 Expression Parsing: {:.2}ms avg ({:.1}ms total)", 
        expr_avg * 1000.0, expr_time.as_secs_f64() * 1000.0);

    // Benchmark complex expressions
    let complex_time = benchmark_complex_expression(iterations, &env)?;
    let complex_avg = complex_time.as_secs_f64() / iterations as f64;
    println!("🧮 Complex Expression: {:.2}ms avg ({:.1}ms total)", 
        complex_avg * 1000.0, complex_time.as_secs_f64() * 1000.0);

    // Performance analysis
    let overhead = (expr_avg / tensor_avg - 1.0) * 100.0;
    let throughput = (width * height) as f64 / (tensor_avg * 1_000_000.0); // Megapixels per second

    println!("");
    println!("📊 Performance Analysis:");
    println!("  Expression parsing overhead: {:.1}%", overhead);
    println!("  Tensor operation throughput: {:.1} Mpx/s", throughput);
    println!("  Memory bandwidth: ~{:.1} GB/s", throughput * 8.0 * 2.0 / 1000.0); // Assuming 2 input tensors

    if overhead < 50.0 {
        println!("  ✅ Expression parsing is highly efficient!");
    } else if overhead < 100.0 {
        println!("  ⚠️ Moderate expression parsing overhead");
    } else {
        println!("  ❌ High expression parsing overhead");
    }

    println!("");
    println!("🎯 Benchmarking complete! Ready for production use.");
    Ok(())
});

⚡ Performance Benchmarking Suite
Grid size: 512x512 = 262144 pixels

Running 10 iterations of each benchmark...
Grid size: 512x512 = 262144 pixels

Running 10 iterations of each benchmark...
🔧 Manual Tensor Ops: 8.87ms avg (88.7ms total)
📝 Expression Parsing: 10.11ms avg (101.1ms total)
🧮 Complex Expression: 25.30ms avg (253.0ms total)

📊 Performance Analysis:
  Expression parsing overhead: 14.1%
  Tensor operation throughput: 29.6 Mpx/s
  Memory bandwidth: ~0.5 GB/s
  ✅ Expression parsing is highly efficient!

🎯 Benchmarking complete! Ready for production use.


In [50]:
// Image Export and Gallery Creation
// Save mathematical visualizations as PNG files for viewing

use std::path::Path;

try_cell!({
    fn save_pattern_as_image(data: &[u8], width: usize, height: usize, filename: &str, colormap: &str) -> Result<()> {
        let img = create_rgb_image(data, width, height, colormap);
        let path = format!("images_store/{}", filename);
        
        // Create directory if it doesn't exist
        if let Some(parent) = Path::new(&path).parent() {
            std::fs::create_dir_all(parent).map_err(|e| anyhow::anyhow!("Failed to create directory: {}", e))?;
        }
        
        img.save(&path).map_err(|e| anyhow::anyhow!("Failed to save image: {}", e))?;
        println!("💾 Saved: {} ({})", filename, colormap);
        Ok(())
    }

    println!("🖼️ Generating Image Gallery...");
    println!("");

    // Save the main patterns with different colormaps
    save_pattern_as_image(&interference_data, width, height, "01_wave_interference_grayscale.png", "grayscale")?;
    save_pattern_as_image(&interference_data, width, height, "01_wave_interference_heat.png", "heat")?;

    save_pattern_as_image(&radial_data, width, height, "02_radial_waves_grayscale.png", "grayscale")?;
    save_pattern_as_image(&radial_data, width, height, "02_radial_waves_ocean.png", "ocean")?;

    save_pattern_as_image(&mandel_data, width, height, "03_mandelbrot_pattern_grayscale.png", "grayscale")?;
    save_pattern_as_image(&mandel_data, width, height, "03_mandelbrot_pattern_heat.png", "heat")?;

    save_pattern_as_image(&flower_data, width, height, "04_flower_pattern_grayscale.png", "grayscale")?;
    save_pattern_as_image(&flower_data, width, height, "04_flower_pattern_ocean.png", "ocean")?;

    // Save additional patterns from previous cells
    let heat_data: Vec<u8> = tensor_to_image(&heat_pattern, "Heat Equation").expect("tensor_to_image failed (heat)");
    save_pattern_as_image(&heat_data, width, height, "05_heat_equation_heat.png", "heat")?;

    let turbulence_data: Vec<u8> = tensor_to_image(&turbulence, "Turbulence").expect("tensor_to_image failed (turbulence)");
    save_pattern_as_image(&turbulence_data, width, height, "06_turbulence_ocean.png", "ocean")?;

    let crystal_data: Vec<u8> = tensor_to_image(&crystal, "Crystalline").expect("tensor_to_image failed (crystal)");
    save_pattern_as_image(&crystal_data, width, height, "07_crystalline_structure_grayscale.png", "grayscale")?;

    let plasma_data: Vec<u8> = tensor_to_image(&plasma, "Plasma").expect("tensor_to_image failed (plasma)");
    save_pattern_as_image(&plasma_data, width, height, "08_plasma_physics_heat.png", "heat")?;

    let quantum_data: Vec<u8> = tensor_to_image(&quantum, "Quantum").expect("tensor_to_image failed (quantum)");
    save_pattern_as_image(&quantum_data, width, height, "09_quantum_wavefunction_ocean.png", "ocean")?;

    let masterpiece_data: Vec<u8> = tensor_to_image(&masterpiece, "Masterpiece").expect("tensor_to_image failed (masterpiece)");
    save_pattern_as_image(&masterpiece_data, width, height, "10_mathematical_masterpiece_heat.png", "heat")?;

    println!("");
    println!("🎨 Gallery Complete!");
    println!("📁 Images saved to: images_store/");
    println!("🖼️ Generated {} image files with multiple colormaps", 15);
    println!("");
    println!("Image List:");
    println!("  • Wave Interference (grayscale + heat colormap)");
    println!("  • Radial Waves (grayscale + ocean colormap)");
    println!("  • Mandelbrot Pattern (grayscale + heat colormap)");
    println!("  • Flower Pattern (grayscale + ocean colormap)");
    println!("  • Heat Equation (heat colormap)");
    println!("  • Turbulence (ocean colormap)");
    println!("  • Crystalline Structure (grayscale)");
    println!("  • Plasma Physics (heat colormap)");
    println!("  • Quantum Wavefunction (ocean colormap)");
    println!("  • Mathematical Masterpiece (heat colormap)");
    Ok(())
});

🖼️ Generating Image Gallery...

💾 Saved: 01_wave_interference_grayscale.png (grayscale)

💾 Saved: 01_wave_interference_grayscale.png (grayscale)
💾 Saved: 01_wave_interference_heat.png (heat)
💾 Saved: 02_radial_waves_grayscale.png (grayscale)
💾 Saved: 02_radial_waves_ocean.png (ocean)
💾 Saved: 03_mandelbrot_pattern_grayscale.png (grayscale)
💾 Saved: 03_mandelbrot_pattern_heat.png (heat)
💾 Saved: 04_flower_pattern_grayscale.png (grayscale)
💾 Saved: 04_flower_pattern_ocean.png (ocean)
🖼️ Converting 'Heat Equation': range [0.000, 1.001]
  ✅ Conversion complete: 262144 pixels
💾 Saved: 05_heat_equation_heat.png (heat)
🖼️ Converting 'Turbulence': range [-1.000, 1.000]
  ✅ Conversion complete: 262144 pixels
💾 Saved: 06_turbulence_ocean.png (ocean)
🖼️ Converting 'Crystalline': range [-1.000, 3.000]
  ✅ Conversion complete: 262144 pixels
💾 Saved: 07_crystalline_structure_grayscale.png (grayscale)
🖼️ Converting 'Plasma': range [-0.965, 0.959]
  ✅ Conversion complete: 262144 pixels
💾 Saved: 08_pla